# Multi-Seed Confirmation — Upstream-Signal Headline

Confirms the single-seed result across seeds 11, 13, 17 (the significance gate before any paper claim). Four conditions per seed, all stock cudalstm:
- **L** baseline
- **L+upQ** upstream OBSERVED Q (oracle ceiling)
- **L+upQ_pred** upstream PREDICTED Q (the realizable headline)
- **L+upQshuf** shuffled-Q null control

Pre-reg `preregistration_multiseed.md`: success = realizable Δ ≥ +0.015, all 3 seeds positive. T4 → Run all (~30–40 min).

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''
SEEDS=[11,13,17]
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; print('Found',c); break
    else: raise RuntimeError('set DRIVE_CAMELS_PATH')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydrology_runs'; os.makedirs(DRIVE_RUNS,exist_ok=True)
print('seeds', SEEDS)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import importlib,sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(os.path.join(REPO_DIR,'runs','topology_ablation','component0'),exist_ok=True)
print('symlinks ready')

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Per-seed sweep (L, L+upQ oracle, L+upQ_pred, L+upQshuf)

For each seed: train L; build the observed-Q oracle feature; train L+upQ; build the
shuffled-Q null; train L+upQshuf; full-span-eval L → predicted Q; build predicted-Q
feature; train L+upQ_pred. Idempotent (skips finished runs).

In [ ]:
%cd {REPO_DIR}
import glob, os
BASIN='topology_analysis/phase1_network_discovery/outputs/component0_basins.txt'
FEAT='experiments/topology_ablation/features'
!python experiments/topology_ablation/generate_topology_attributes.py 2>&1 | tail -1
for SEED in SEEDS:
    print(f'\n############ SEED {SEED} ############')
    Ldir=f'{REPO_DIR}/runs/topology_ablation/component0/L_component0_seed{SEED}'
    # L baseline
    if not os.path.isfile(f'{Ldir}/test/model_epoch030/test_metrics.csv'):
        !python experiments/topology_ablation/make_configs.py --network component0 --basin-file {BASIN} --seed {SEED} --device cuda:0 --epochs 30
        !python neuralhydrology/nh_run.py train --config-file experiments/topology_ablation/configs/L_component0_seed{SEED}.yaml 2>&1 | tail -2
        ts=sorted(glob.glob(f'{Ldir}_*'))
        if ts: os.rename(ts[-1], Ldir)
        !python neuralhydrology/nh_run.py evaluate --run-dir {Ldir} --epoch 30 2>&1 | tail -1
    # observed-Q oracle
    !python experiments/topology_ablation/build_upstream_discharge_feature.py --network component0 --lag-days 1 2>&1 | tail -1
    !python experiments/topology_ablation/run_upstream_feature.py --network component0 --seed {SEED} --device cuda:0 --feature-file {FEAT}/upstream_q_component0_lag1.p --cond-name L_upQ 2>&1 | tail -2
    # shuffled-Q null
    !python experiments/topology_ablation/build_upstream_variants.py --network component0 --variant shuffled --lag-days 1 --seed {SEED} 2>&1 | tail -1
    !python experiments/topology_ablation/run_upstream_feature.py --network component0 --seed {SEED} --device cuda:0 --feature-file {FEAT}/upstream_q_shuffled_component0_lag1.p --cond-name L_upQshuf 2>&1 | tail -2
    # predicted-Q realizable (per-seed full-span eval, clean --seed arg)
    !python experiments/topology_ablation/build_predicted_upstream_q.py --network component0 --seed {SEED} --lag-days 1 2>&1 | tail -1
    !python experiments/topology_ablation/run_upstream_feature.py --network component0 --seed {SEED} --device cuda:0 --feature-file {FEAT}/upstream_q_pred_component0_seed{SEED}_lag1.p --cond-name L_upQpred 2>&1 | tail -2
    print(f'seed {SEED} done')

## Cell 8 — Multi-seed verdict (mean ± std, realizable gain vs ceiling)

In [ ]:
%cd {REPO_DIR}
import pandas as pd, numpy as np, os
base=f'{REPO_DIR}/runs/topology_ablation/component0'
def nse(c,s):
    p=f'{base}/{c}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.exists(p) else None
rows=[]
for c in ['L','L_upQ','L_upQpred','L_upQshuf']:
    meds=[nse(c,s).median() for s in SEEDS if nse(c,s) is not None]
    if meds: rows.append((c, np.mean(meds), np.std(meds), meds))
print('condition       mean±std median NSE     per-seed')
for c,m,sd,meds in rows:
    print(f'{c:<12} {m:+.4f} ± {sd:.4f}   {[round(x,3) for x in meds]}')
# realizable headline: paired Δ per seed
print('\n=== L+upQ_pred − L (realizable), paired per seed ===')
deltas=[]
for s in SEEDS:
    L=nse('L',s); P=nse('L_upQpred',s)
    if L is None or P is None: continue
    b=L.index.intersection(P.index); d=(P.loc[b]-L.loc[b]).dropna(); deltas.append(d.median())
    print(f'  seed {s}: median Δ {d.median():+.4f}  frac>0 {(d>0).mean():.2f}')
if deltas:
    print(f'\ncross-seed mean Δ: {np.mean(deltas):+.4f} ± {np.std(deltas):.4f}  (all positive: {all(x>0 for x in deltas)})')
    v='SUCCESS (publishable)' if (np.mean(deltas)>=0.015 and all(x>0 for x in deltas)) else 'CHECK — see pre-reg'
    print(f'PRE-REG VERDICT: {v}')

## Done
Pull the verdict + run folders, then `crs interpret multiseed`. If success → the headline has 3-seed support; proceed to writing / scale curve.